In [1]:
#Primero hay que descargar librerias, instalar Apache Cassandra y configurar para realizar las consultas.

#Todo lo que tiene que ver con la tecnologia cassandra
from cassandra.cluster import Cluster
from cassandra.query import dict_factory 

#Pandas para analisis y manejo de datos 
import pandas as pd

#numpy para operaciones numericas y generacion de rangos o vectores
import numpy as np

#datetime para fechas y tiempos
from datetime import datetime 


In [2]:
#Ahora que ya se descargaron las librerias, se realiza la conexion entre python apache cassandra.

#Realizar la conexion cassandra mediante el driver pyhton.
cluster=Cluster(['127.0.0.1'])
session=cluster.connect()

#definir el formato, este formato devuelve las consultas como diccionario.
session.row_factory =dict_factory

#definir el nombre del keyspace (el keyspace es como la base donde se guardan todas las tablas que hagamos)
keyspace_nombre ='aquasense'


session.set_keyspace(keyspace_nombre)




In [3]:
#Consultas por realizar: 

#1. Ultimas lecturas de un sensor (ultimas 10):
def ultimas_lecturas_sensor(sensor_id, bucket):
    query="""
    SELECT *
    FROM lecturas_por_sensor
    WHERE sensor_id = %s 
    AND bucket = %s
   LIMIT 10  
   """
    rows = session.execute(query, (sensor_id, bucket))
    return list(rows)

In [4]:
#2. Lecturas por rango temporal

def lecturas_por_rango(sensor_id, bucket, ts_inicio, ts_fin):
    query = """
    SELECT *
    FROM lectura_por_sensor
    WHERE sensor_id = %s 
    AND bucket = %s
    AND fecha_hora >= %s 
    AND fecha_hora <= %s
    """
    rows = session.execute(query, (sensor_id, bucket, ts_inicio, ts_fin))
    return list(rows)

In [5]:
#3. Consulta por zona y periodo
def lecturas_por_zona_periodo(zona, bucket, ts_inicio, ts_fin):
    query = """
    SELECT *
    FROM lecturas_por_zona
    WHERE zona = %s 
    AND bucket = %s
    AND fecha_hora >= %s 
    AND fecha_hora <= %s
    """
    rows = session.execute(query, (zona, bucket, ts_inicio, ts_fin))
    return list(rows)

In [6]:
#4. Identificar anomalias: 
def anomalias_sensor(sensor_id, bucket):
    query = """
    SELECT *
    FROM anomalias_por_sensor
    WHERE sensor_id = %s
    AND bucket = %s
    """
    rows = session.execute(query, (sensor_id, bucket))
    return list(rows)


In [7]:
#5. La consulta optimizada escogida es un resumen diario por zona

def resumen_diario (zona):
    query = """
    SELECT *
    FROM resumen_diario_zona
    WHERE zona = %s
    """
    rows = session.execute(query, (zona,))
    return list(rows)

In [8]:
#6. Extra. La ultima lectura de un sensor (la mas reciente), esta se escoge porque existe la tabla sensor_latest:
def ultima_lectura(sensor_id):
    query = """
    SELECT *
    FROM ultima_lectura_sensor
    WHERE sensor_id = %s
    """
    rows = session.execute(query, (sensor_id,))
    return rows.one()